# BFF: self-replicators from a soup of random programs

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxencefaldor/cax/blob/main/examples/70_bff.ipynb)

BFF is the Brainfuck dialect of *Computational Life: How Well-formed, Self-replicating Programs Emerge from Simple Interaction* (Agüera y Arcas et al., 2024). A soup holds thousands of 64-byte strings. Each string is a program **and** its own memory: the instruction pointer and the two data heads all address the same tape, so a program rewrites itself as it runs. Every epoch the strings are paired at random, each pair is concatenated into one 128-byte tape and executed for a fixed budget of steps, and the two halves go back into the soup. There is no fitness function and nothing is copied on purpose. Self-replicators appear anyway, and once one does it takes over the soup within a few hundred epochs.

The language has ten instructions; every other byte is inert. The heads `head0` (read) and `head1` (write) move with `< >` and `{ }`, `+ -` change the byte under `head0`, `. ,` copy a byte between the heads, and `[ ]` loop on the byte under `head0`, as in Brainfuck.

The implementation follows the reference `cubff` code byte for byte (the `bff_noheads` language of the paper's Section 2). See `notes/` on the `bff` branch for the verification.

In [ ]:
%pip install -q cax brotli

In [ ]:
import jax
import jax.numpy as jnp
import mediapy
import numpy as np
from flax import nnx

from cax.cs.bff import (
    BFF,
    BFFGrid,
    high_order_entropy,
    parse,
    replication_score,
    run,
    sample_partners,
    skeleton_hash,
    unparse,
)

## A self-replicator by hand

Figure 4 of the paper shows a 17-byte replicator. It is a palindrome: the read head walks right and the write head walks left, so the copy comes out reversed, which is the same string. Put it at the start of a tape whose second half is zeros and run it for the reference budget of 8192 steps.

In [ ]:
program = "[[{.>]-]A]-]>.{[["  # `A` is any inert byte
tape = jnp.zeros((128,), dtype=jnp.uint8).at[: len(program)].set(parse(program))

tapes, steps, ops = run(tape[None], BFF(rngs=nnx.Rngs(0)).opcode_table)
print("first half :", unparse(tapes[0, :64]))
print("second half:", unparse(tapes[0, 64:]))
print("steps", int(steps[0]), "instructions executed", int(ops[0]))

The replication detector of the reference code scores a program by how many of its bytes it reproduces across chains of executions against random partners. The hand-written replicator reproduces exactly its own 17 bytes; a random program reproduces none.

In [ ]:
cs = BFF(rngs=nnx.Rngs(0))
random_program = jax.random.randint(jax.random.key(1), (64,), 0, 256, dtype=jnp.uint8)
programs = jnp.stack([tape[:64], random_program])
partners = sample_partners(jax.random.key(2), programs.shape[0])
print(replication_score(programs, partners, cs.opcode_table))

## Seeding a soup

The state of the system is the soup itself, an array of shape `(num_programs, 64)`. One step of the system is one epoch. Seed a random soup with a single copy of the replicator and watch the paper's complexity signal, the *high-order entropy* (byte entropy minus compressed size in bits per byte), which is near zero for random bytes and jumps when the soup fills with copies of one string. In the paper, a single seeded replicator takes over about a fifth of the time; try a few seeds.

In [ ]:
num_programs = 4096
cs = BFF(rngs=nnx.Rngs(3))
soup = cs.init_state(num_programs=num_programs)
soup = soup.at[0, : len(program)].set(parse(program))

history = [soup]
for _ in range(16):
    soup = cs(soup, num_steps=8)
    history.append(soup)
complexity = [high_order_entropy(np.asarray(s)) for s in history]
print([round(c, 2) for c in complexity])

In [ ]:
frames = [cs.render(s[:512]) for s in history]
mediapy.show_images(
    [np.asarray(f) for f in frames[::4]],
    titles=[f"epoch {8 * i}" for i in range(0, len(frames), 4)],
    height=256,
)

Green is a bracket, magenta a tape write, lilac a head move, red the zero byte, grey anything inert. Rows are programs; once the replicator has spread, the rows become copies of one another.

## Emergence from a random soup

Without seeding, replicators must arise on their own. In the paper this happens in about 40% of runs of 2^17 programs within 16k epochs, which is a billion pair executions. This notebook does not attempt that; the run below is a small soup for a few epochs, to show the loop. The `bff` branch notes report the reduced-scale reproduction and how long it takes on a CPU.

In [ ]:
cs = BFF(rngs=nnx.Rngs(0))
soup = cs.init_state(num_programs=2048)
soup, soups = cs(soup, num_steps=64, return_states=True)
print("high-order entropy after 64 epochs:", high_order_entropy(np.asarray(soup)))

## Space: programs on a torus

With `grid=(height, width)` the soup is a torus and each epoch pairs every program with one of its four neighbours instead of a random partner. Interaction becomes geometric, and the render changes: one pixel per program, coloured by its *species*, the hash of its instruction skeleton, and brightened by its instruction density. A random soup is dark noise; a replicator's colony is a bright patch. Plant the replicator in the centre and watch it spread; as in the paper's seeded runs, the seed takes only about one time in four, so the seed below was chosen to be one that does.

In [ ]:
height, width = 32, 64
cs = BFF(grid=(height, width), rngs=nnx.Rngs(11))
soup = cs.init_state()
centre = (height // 2) * width + width // 2
soup = soup.at[centre, : len(program)].set(parse(program))

frames, epochs = [cs.render(soup)], [0]
for i in range(8):
    soup = cs(soup, num_steps=48)
    frames.append(cs.render(soup))
    epochs.append(48 * (i + 1))
    species = np.unique(np.asarray(skeleton_hash(soup, cs.opcode_table))).size
    entropy = high_order_entropy(np.asarray(soup))
    print(f"epoch {epochs[-1]:4d}: high-order entropy {entropy:.2f}, species {species}")
mediapy.show_images(
    [np.asarray(f) for f in frames[::2]],
    titles=[f"epoch {e}" for e in epochs[::2]],
    height=160,
)

The colony grows as a disc with a ragged front and, inside, splinters into many colours: mutants of the replicator that coexist as patches. Under random pairing the same takeover has no geography, and one lineage dominates.

## The grid machine

`BFFGrid` drops the tapes altogether. Memory is one byte array on a torus of any dimension, and threads walk it: each step a thread reads the byte under its pointer, executes it, and moves one cell in its direction. Brackets turn instead of jumping, so in one dimension a loop is the pointer bouncing between two brackets and in two dimensions it is a closed rectilinear path. Every instruction is total; a thread lives a fixed number of steps and respawns elsewhere. The state is literally the picture: instructions in colour, data in grey, threads in white.

In [ ]:
cs = BFFGrid(rngs=nnx.Rngs(0))
state = cs.init_state(shape=(64, 128))
frames = [cs.render(state)]
for _ in range(3):
    state = cs(state, num_steps=2048)
    frames.append(cs.render(state))
mediapy.show_images(
    [np.asarray(f) for f in frames],
    titles=[f"step {2048 * i}" for i in range(4)],
    height=192,
)

Threads leave trails of writes behind them. Whether self-replicators emerge in this machine, and how the pictures look when they do, is the open question of the `bff` branch notes.